# Shared Next-POI GRU With Candidate Masks

This notebook prepares one next-POI GRU for both `/recommend` and `/generate-course`.
It intentionally stops before full training: data prep, POI Master, model vocabulary, candidate masks, dataloaders, model code, API simulators, and preflight checks are ready. The final training cell is guarded by `RUN_FULL_TRAINING = False`.


In [ ]:
from __future__ import annotations

import json
import math
import pickle
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset

SEED = 42
ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = ROOT / "data" / "processed" / "final"
ARTIFACT_DIR = ROOT / "artifacts" / "shared_next_poi_gru_experiment"
TRAVELER_PATH = DATA_DIR / "final_traveler_features.csv"
TRAVEL_PATH = DATA_DIR / "final_travel_features.csv"
SEQUENCE_PATH = DATA_DIR / "final_travel_sequence.csv"

COURSE_POIS_PER_DAY = 6
MIN_SEQUENCE_LEN = 2
MAX_SEQUENCE_LEN = 20
TOP_KS = (1, 3, 5, 10)
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
SPECIAL_TOKEN_IDS = {0, 1}
AREA_CODE_TO_NAME = {
    "11000": "서울", "41110": "수원", "28000": "인천", "30000": "대전",
    "27000": "대구", "12000": "광주", "26000": "부산", "48120": "창원",
}
AREA_NAME_TO_CODE = {name: code for code, name in AREA_CODE_TO_NAME.items()}

def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


## 1. Load CSVs, Build Ordered Sequences, And Build POI Master

`final_travel_sequence.csv` is treated as the service-supported ordered subsequence. POIs outside the supported backend area-name mapping are excluded from model candidates and training sequences.


In [ ]:
traveler_df = pd.read_csv(TRAVELER_PATH, encoding="utf-8-sig")
travel_df = pd.read_csv(TRAVEL_PATH, encoding="utf-8-sig")
sequence_df = pd.read_csv(SEQUENCE_PATH, encoding="utf-8-sig")

required = {
    "traveler": {"traveler_id", "residence_code", "gender", "age_grp", "travel_style", "travel_like_code"},
    "travel": {"travel_id", "traveler_id", "companion_count", "theme"},
    "sequence": {"travel_id", "day_index", "visit_order", "visit_area_nm", "content_id", "area"},
}
assert required["traveler"].issubset(traveler_df.columns)
assert required["travel"].issubset(travel_df.columns)
assert required["sequence"].issubset(sequence_df.columns)

def normalize_content_id_value(value: Any) -> str:
    if pd.isna(value):
        return ""
    text = str(value).strip()
    return text[:-2] if text.endswith(".0") else text

def mode_or_first(values: pd.Series) -> str:
    clean = values.dropna().astype(str)
    if clean.empty:
        return ""
    modes = clean.mode()
    return str(modes.iloc[0] if not modes.empty else clean.iloc[0])

seq = sequence_df.copy()
seq["content_id"] = seq["content_id"].map(normalize_content_id_value)
seq["area_name"] = seq["area"].astype(str).str.strip()
seq["content_name"] = seq["visit_area_nm"].astype(str).str.strip()
seq["area_code"] = seq["area_name"].map(AREA_NAME_TO_CODE)
seq["day_index"] = pd.to_numeric(seq["day_index"], errors="coerce").fillna(1).astype(int)
seq["visit_order"] = pd.to_numeric(seq["visit_order"], errors="coerce").fillna(1).astype(int)
seq = seq.loc[seq["content_id"].ne("")].sort_values(["travel_id", "day_index", "visit_order"], kind="mergesort")
seq_supported = seq.loc[seq["area_code"].notna()].copy()

poi_master_base = (
    seq_supported.groupby("content_id", as_index=False)
    .agg(
        area_code=("area_code", mode_or_first),
        area_name=("area_name", mode_or_first),
        content_name=("content_name", mode_or_first),
        visit_count=("travel_id", "size"),
    )
    .sort_values(["area_code", "content_id"], kind="mergesort")
    .reset_index(drop=True)
)
trip_sequences = (
    seq_supported.groupby("travel_id", sort=False)
    .agg(content_sequence=("content_id", list), area_sequence=("area_code", list))
    .reset_index()
)
trip_sequences["sequence_len"] = trip_sequences["content_sequence"].map(len)
trip_features = travel_df.merge(traveler_df, on="traveler_id", how="inner", validate="many_to_one")
model_input = trip_features.merge(trip_sequences, on="travel_id", how="inner", validate="one_to_one")
model_input = model_input.loc[model_input["sequence_len"] >= MIN_SEQUENCE_LEN].reset_index(drop=True)
coverage = {
    "traveler_rows": len(traveler_df), "travel_rows": len(travel_df), "sequence_rows": len(sequence_df),
    "supported_sequence_rows": len(seq_supported), "poi_master_rows_before_vocab": len(poi_master_base),
    "model_input_rows": len(model_input), "sequence_content_unique": sequence_df["content_id"].nunique(),
}
print(json.dumps(coverage, ensure_ascii=False, indent=2))
print(model_input["sequence_len"].describe())
poi_master_base.head()


## 2. Split, Vocabulary, Feature Encoder, And Samples

The model vocabulary is built from train-trip POIs. Inference prefixes project to this vocabulary by dropping unknown POIs, never by converting unknown POIs to `<UNK>`.


In [ ]:
train_trips, temp_trips = train_test_split(model_input["travel_id"], test_size=0.30, random_state=SEED, shuffle=True)
valid_trips, test_trips = train_test_split(temp_trips, test_size=0.50, random_state=SEED, shuffle=True)
train_df = model_input.loc[model_input["travel_id"].isin(set(train_trips))].reset_index(drop=True)
valid_df = model_input.loc[model_input["travel_id"].isin(set(valid_trips))].reset_index(drop=True)
test_df = model_input.loc[model_input["travel_id"].isin(set(test_trips))].reset_index(drop=True)

train_content_ids = sorted({str(cid) for s in train_df["content_sequence"] for cid in s})
content_id_to_token = {PAD_TOKEN: 0, UNK_TOKEN: 1}
content_id_to_token.update({cid: idx + 2 for idx, cid in enumerate(train_content_ids)})
token_to_content_id = {tid: cid for cid, tid in content_id_to_token.items()}

poi_master = poi_master_base.copy()
poi_master["token_id"] = poi_master["content_id"].map(content_id_to_token)
poi_master["is_model_candidate"] = poi_master["token_id"].notna()
poi_master["token_id"] = poi_master["token_id"].astype("Int64")
model_poi_master = poi_master.loc[poi_master["is_model_candidate"]].copy()

FEATURE_COLUMNS = ["companion_count", "theme", "residence_code", "gender", "age_grp", "travel_style", "travel_like_code"]
NUMERIC_FEATURES = ["companion_count", "age_grp"]
CATEGORICAL_FEATURES = [c for c in FEATURE_COLUMNS if c not in NUMERIC_FEATURES]
feature_encoder = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median", keep_empty_features=True)), ("scaler", StandardScaler())]), NUMERIC_FEATURES),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="missing", keep_empty_features=True)), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)
feature_encoder.fit(train_df[FEATURE_COLUMNS])
USER_FEATURE_DIM = int(feature_encoder.transform(train_df[FEATURE_COLUMNS].head(1)).shape[1])

def project_known_sequence(content_id_sequence: Iterable[Any]) -> list[str]:
    known = []
    for value in content_id_sequence:
        cid = normalize_content_id_value(value)
        if cid in content_id_to_token and cid not in {PAD_TOKEN, UNK_TOKEN}:
            known.append(cid)
    return known

def tokens_for_known_sequence(content_id_sequence: Iterable[Any]) -> list[int]:
    return [int(content_id_to_token[cid]) for cid in project_known_sequence(content_id_sequence)]

def make_next_poi_samples(df: pd.DataFrame) -> list[dict[str, Any]]:
    encoded = feature_encoder.transform(df[FEATURE_COLUMNS]).astype(np.float32)
    samples = []
    for row_idx, row in df.reset_index(drop=True).iterrows():
        content_sequence = [normalize_content_id_value(v) for v in row["content_sequence"]]
        for target_pos in range(1, len(content_sequence)):
            label_cid = content_sequence[target_pos]
            if label_cid not in content_id_to_token:
                continue
            known_prefix = project_known_sequence(content_sequence[:target_pos])[-MAX_SEQUENCE_LEN:]
            if not known_prefix:
                continue
            samples.append({
                "travel_id": row["travel_id"], "user_features": encoded[row_idx],
                "prefix": [content_id_to_token[cid] for cid in known_prefix],
                "label": int(content_id_to_token[label_cid]), "label_content_id": label_cid,
            })
    return samples

train_samples = make_next_poi_samples(train_df)
valid_samples = make_next_poi_samples(valid_df)
test_samples = make_next_poi_samples(test_df)
split_sizes = {"train": len(train_df), "valid": len(valid_df), "test": len(test_df)}
sample_sizes = {"train": len(train_samples), "valid": len(valid_samples), "test": len(test_samples)}
print("split_sizes", split_sizes)
print("sample_sizes", sample_sizes)
print("vocab_size", len(content_id_to_token), "candidate_pois", len(model_poi_master), "feature_dim", USER_FEATURE_DIM)
assert train_samples, "No train samples were created."


## 3. Dataset, Metrics, Baseline, And GRU Model


In [ ]:
class TravelSequenceDataset(Dataset):
    def __init__(self, samples: list[dict[str, Any]]) -> None:
        self.samples = samples
    def __len__(self) -> int:
        return len(self.samples)
    def __getitem__(self, idx: int) -> dict[str, Any]:
        return self.samples[idx]

def collate_batch(batch: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
    user_features = torch.tensor(np.stack([x["user_features"] for x in batch]), dtype=torch.float32)
    lengths = torch.tensor([len(x["prefix"]) for x in batch], dtype=torch.long)
    sequences = torch.zeros((len(batch), int(lengths.max().item())), dtype=torch.long)
    for idx, item in enumerate(batch):
        sequences[idx, : len(item["prefix"])] = torch.tensor(item["prefix"], dtype=torch.long)
    labels = torch.tensor([x["label"] for x in batch], dtype=torch.long)
    return {"user_features": user_features, "sequences": sequences, "lengths": lengths, "labels": labels}

def make_loader(samples: list[dict[str, Any]], batch_size: int, shuffle: bool) -> DataLoader:
    return DataLoader(TravelSequenceDataset(samples), batch_size=batch_size, shuffle=shuffle, collate_fn=collate_batch)

def metric_at_k(labels: np.ndarray, scores: np.ndarray, ks: tuple[int, ...] = TOP_KS) -> dict[str, float]:
    order = np.argsort(-scores, axis=1)
    metrics = {}
    for k in ks:
        topk = order[:, : min(k, scores.shape[1])]
        hits = topk == labels[:, None]
        metrics[f"recall@{k}"] = float(hits.any(axis=1).mean())
        rr, ndcg = [], []
        for row_hits in hits:
            pos = np.flatnonzero(row_hits)
            rr.append(0.0 if len(pos) == 0 else 1.0 / float(pos[0] + 1))
            ndcg.append(0.0 if len(pos) == 0 else 1.0 / math.log2(float(pos[0] + 2)))
        metrics[f"mrr@{k}"] = float(np.mean(rr))
        metrics[f"ndcg@{k}"] = float(np.mean(ndcg))
    return metrics

def baseline_scores(train_items: list[dict[str, Any]], eval_items: list[dict[str, Any]], vocab_size: int) -> np.ndarray:
    counts = np.ones(vocab_size, dtype=np.float32) * 1e-6
    counts[list(SPECIAL_TOKEN_IDS)] = -np.inf
    for item in train_items:
        counts[int(item["label"])] += 1.0
    return np.repeat(counts[None, :], repeats=len(eval_items), axis=0)

@dataclass(frozen=True)
class GRUContentConfig:
    embedding_dim: int = 16
    hidden_dim: int = 64
    dropout: float = 0.2
    learning_rate: float = 1e-3
    batch_size: int = 64
    epochs: int = 8
    weight_decay: float = 1e-5

class GRUContentRecommender(nn.Module):
    def __init__(self, vocab_size: int, user_feature_dim: int, embedding_dim: int, hidden_dim: int, dropout: float) -> None:
        super().__init__()
        self.place_embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.user_projection = nn.Sequential(nn.Linear(user_feature_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout))
        self.gru = nn.GRU(input_size=embedding_dim, hidden_size=hidden_dim, batch_first=True)
        self.output = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_dim * 2, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, vocab_size))
    def forward(self, user_features: torch.Tensor, sequences: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.place_embedding(sequences)
        gru_output, _ = self.gru(embedded)
        last_idx = torch.clamp(lengths - 1, min=0)
        gather_idx = last_idx.view(-1, 1, 1).expand(-1, 1, gru_output.size(2))
        sequence_context = torch.gather(gru_output, dim=1, index=gather_idx).squeeze(1)
        logits = self.output(torch.cat([sequence_context, self.user_projection(user_features)], dim=1))
        logits[:, list(SPECIAL_TOKEN_IDS)] = -1e9
        return logits

valid_labels = np.asarray([x["label"] for x in valid_samples], dtype=np.int64)
baseline_valid = metric_at_k(valid_labels, baseline_scores(train_samples, valid_samples, len(content_id_to_token)))
model = GRUContentRecommender(len(content_id_to_token), USER_FEATURE_DIM, 16, 64, 0.2).to(DEVICE)
batch = next(iter(make_loader(train_samples[:8], batch_size=4, shuffle=False)))
with torch.no_grad():
    shape = model(batch["user_features"].to(DEVICE), batch["sequences"].to(DEVICE), batch["lengths"].to(DEVICE)).shape
assert shape == (4, len(content_id_to_token))
baseline_valid


## 4. Common Candidate Constraints And API Simulators

Both simulated APIs use the same POI Master, area mask, valid-POI mask, and duplicate mask.


In [ ]:
token_area_code = np.full(len(content_id_to_token), "", dtype=object)
valid_token_mask = np.zeros(len(content_id_to_token), dtype=bool)
content_id_to_area_code: dict[str, str] = {}
for row in model_poi_master.itertuples(index=False):
    tid = int(row.token_id)
    token_area_code[tid] = str(row.area_code)
    valid_token_mask[tid] = True
    content_id_to_area_code[str(row.content_id)] = str(row.area_code)
valid_token_mask[list(SPECIAL_TOKEN_IDS)] = False

def area_token_mask(area_code: str) -> np.ndarray:
    return valid_token_mask & (token_area_code == str(area_code).strip())

def apply_candidate_mask(logits: np.ndarray | torch.Tensor, area_code: str, excluded_content_ids: Iterable[Any] = ()) -> np.ndarray:
    scores = logits.detach().cpu().numpy().astype(np.float32).copy() if isinstance(logits, torch.Tensor) else np.asarray(logits, dtype=np.float32).copy()
    if scores.ndim != 1 or scores.shape[0] != len(content_id_to_token):
        raise ValueError("logits must be a 1D vector matching the model vocabulary size")
    scores[~area_token_mask(area_code)] = -np.inf
    scores[list(SPECIAL_TOKEN_IDS)] = -np.inf
    for value in excluded_content_ids:
        token_id = content_id_to_token.get(normalize_content_id_value(value))
        if token_id is not None:
            scores[int(token_id)] = -np.inf
    return scores

def top_content_ids_from_scores(scores: np.ndarray, top_k: int) -> list[dict[str, Any]]:
    finite = np.flatnonzero(np.isfinite(scores))
    order = finite[np.argsort(-scores[finite])]
    return [{"content_id": token_to_content_id[int(tid)], "token_id": int(tid), "score": float(scores[int(tid)])} for tid in order[:top_k]]

def fallback_top_k(area_code: str, excluded_content_ids: Iterable[Any] = (), top_k: int = 4) -> list[dict[str, Any]]:
    excluded = {normalize_content_id_value(v) for v in excluded_content_ids}
    rows = model_poi_master.loc[model_poi_master["area_code"].eq(str(area_code).strip())]
    rows = rows.sort_values(["visit_count", "content_id"], ascending=[False, True], kind="mergesort")
    rows = rows.loc[~rows["content_id"].isin(excluded)].head(top_k)
    return [{"content_id": str(r.content_id), "token_id": int(r.token_id), "score": 1.0 - rank * 0.05} for rank, r in enumerate(rows.itertuples(index=False))]

def encode_feature_request(request: dict[str, Any]) -> np.ndarray:
    row = {column: request.get(column) for column in FEATURE_COLUMNS}
    row["theme"] = request.get("travelPersona", row.get("theme"))
    row["age_grp"] = request.get("ageGroup", row.get("age_grp"))
    row["travel_style"] = request.get("travelerStyle", row.get("travel_style"))
    row["residence_code"] = request.get("residenceArea", row.get("residence_code"))
    row["companion_count"] = request.get("companionCount", row.get("companion_count"))
    return feature_encoder.transform(pd.DataFrame([row], columns=FEATURE_COLUMNS)).astype(np.float32)

def logits_for_prefix(model_obj: nn.Module, request: dict[str, Any], prefix_content_ids: list[str]) -> np.ndarray:
    prefix_tokens = tokens_for_known_sequence(prefix_content_ids)[-MAX_SEQUENCE_LEN:]
    if not prefix_tokens:
        raise ValueError("known prefix is empty")
    model_obj.eval()
    with torch.no_grad():
        logits = model_obj(
            torch.tensor(encode_feature_request(request), dtype=torch.float32, device=DEVICE),
            torch.tensor([prefix_tokens], dtype=torch.long, device=DEVICE),
            torch.tensor([len(prefix_tokens)], dtype=torch.long, device=DEVICE),
        )[0]
    return logits.detach().cpu().numpy()

def recommend_simulation(model_obj: nn.Module, request: dict[str, Any], top_k: int = 4) -> dict[str, Any]:
    area_code = str(request["areaCode"]).strip()
    original_sequence = [normalize_content_id_value(v) for v in request.get("contentIdSequence", [])]
    target_length = int(request["travelDuration"]) * COURSE_POIS_PER_DAY
    if not original_sequence:
        return {"mode": "fallback", "reason": "empty_sequence", "recommendations": fallback_top_k(area_code, [], top_k)}
    if len(original_sequence) >= target_length:
        return {"mode": "fallback", "reason": "full_course_sequence", "recommendations": fallback_top_k(area_code, original_sequence, top_k)}
    known_sequence = project_known_sequence(original_sequence)
    if not known_sequence:
        return {"mode": "fallback", "reason": "empty_known_sequence", "recommendations": fallback_top_k(area_code, original_sequence, top_k)}
    scores = apply_candidate_mask(logits_for_prefix(model_obj, request, known_sequence), area_code, original_sequence)
    recs = top_content_ids_from_scores(scores, top_k)
    if len(recs) < top_k:
        recs.extend(fallback_top_k(area_code, [*original_sequence, *[x["content_id"] for x in recs]], top_k - len(recs)))
    return {"mode": "gru", "known_sequence": known_sequence, "recommendations": recs[:top_k]}

def validate_required_content_ids(area_code: str, content_id_list: Iterable[Any], target_length: int) -> list[str]:
    required, seen = [], set()
    for value in content_id_list:
        cid = normalize_content_id_value(value)
        if cid in seen:
            continue
        seen.add(cid)
        if cid not in content_id_to_token or cid not in content_id_to_area_code:
            raise ValueError(f"contentIdList contains a POI outside the model candidate set: {cid}")
        if content_id_to_area_code[cid] != str(area_code).strip():
            raise ValueError(f"contentIdList contains a POI outside areaCode {area_code}: {cid}")
        required.append(cid)
    if len(required) > target_length:
        raise ValueError("contentIdList cannot contain more unique items than target course length")
    return required

def generate_course_simulation(model_obj: nn.Module, request: dict[str, Any]) -> dict[str, Any]:
    area_code = str(request["areaCode"]).strip()
    target_length = int(request["travelDuration"]) * COURSE_POIS_PER_DAY
    required = validate_required_content_ids(area_code, request.get("contentIdList", []), target_length)
    generated = [required[0]] if required else [fallback_top_k(area_code, [], 1)[0]["content_id"]]
    while len(generated) < target_length:
        missing_required = [cid for cid in required if cid not in generated]
        if missing_required and len(missing_required) >= target_length - len(generated):
            generated.append(missing_required[0])
            continue
        scores = apply_candidate_mask(logits_for_prefix(model_obj, request, generated), area_code, generated)
        next_items = top_content_ids_from_scores(scores, 1) or fallback_top_k(area_code, generated, 1)
        if not next_items:
            raise ValueError("No valid content_id candidates remain for course generation")
        generated.append(next_items[0]["content_id"])
    return {"mode": "autoregressive_gru", "target_length": target_length, "required_content_ids": required, "content_id_sequence": generated}


## 5. Preflight Checks Before Training


In [ ]:
class DummyNextPoiModel(nn.Module):
    def __init__(self, vocab_size: int) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        self.call_count = 0
    def forward(self, user_features: torch.Tensor, sequences: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        self.call_count += 1
        return torch.arange(self.vocab_size, dtype=torch.float32, device=user_features.device).repeat(user_features.size(0), 1)

assert len(traveler_df) > 0 and len(travel_df) > 0 and len(sequence_df) > 0
assert len(model_input) > 0 and len(model_poi_master) > 0
sample_cid = str(model_poi_master.iloc[0]["content_id"])
assert token_to_content_id[int(content_id_to_token[sample_cid])] == sample_cid

area_code = "11000"
area_candidates = model_poi_master.loc[model_poi_master["area_code"].eq(area_code), "content_id"].astype(str).tolist()
if len(area_candidates) < COURSE_POIS_PER_DAY:
    area_code = str(model_poi_master["area_code"].value_counts().index[0])
    area_candidates = model_poi_master.loc[model_poi_master["area_code"].eq(area_code), "content_id"].astype(str).tolist()
assert len(area_candidates) >= COURSE_POIS_PER_DAY
mask = area_token_mask(area_code)
assert mask.sum() == len(model_poi_master.loc[model_poi_master["area_code"].eq(area_code)])
assert not mask[list(SPECIAL_TOKEN_IDS)].any()

known_a, known_b = area_candidates[:2]
unknown = "999999999999"
assert project_known_sequence([known_a, unknown, known_b]) == [known_a, known_b]
base_row = model_input.iloc[0].to_dict()
base_request = {column: base_row.get(column) for column in FEATURE_COLUMNS}
base_request.update({"areaCode": area_code, "travelDuration": "1"})

dummy = DummyNextPoiModel(len(content_id_to_token)).to(DEVICE)
unknown_response = recommend_simulation(dummy, {**base_request, "contentIdSequence": [unknown]}, top_k=4)
assert unknown_response["mode"] == "fallback" and unknown_response["reason"] == "empty_known_sequence"
assert dummy.call_count == 0

gru_response = recommend_simulation(dummy, {**base_request, "contentIdSequence": [known_a, unknown, known_b]}, top_k=4)
assert gru_response["mode"] == "gru" and dummy.call_count == 1
recommended_ids = [x["content_id"] for x in gru_response["recommendations"]]
assert known_a not in recommended_ids and known_b not in recommended_ids
assert all(content_id_to_area_code[cid] == area_code for cid in recommended_ids)

course_response = generate_course_simulation(dummy, {**base_request, "contentIdList": area_candidates[:2]})
course_ids = course_response["content_id_sequence"]
assert len(course_ids) == COURSE_POIS_PER_DAY
assert len(course_ids) == len(set(course_ids))
assert set(area_candidates[:2]).issubset(course_ids)
assert all(content_id_to_area_code[cid] == area_code for cid in course_ids)

print("Preflight checks passed.")
print("recommend sample", gru_response)
print("course sample", course_response)


## 6. Training Code Prepared But Not Executed

Set `RUN_FULL_TRAINING = True` only when ready to run the actual experiment.


In [ ]:
def run_epoch(model_obj: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer | None = None) -> float:
    is_train = optimizer is not None
    model_obj.train(is_train)
    loss_fn = nn.CrossEntropyLoss()
    total_loss, total_rows = 0.0, 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.set_grad_enabled(is_train):
            logits = model_obj(batch["user_features"], batch["sequences"], batch["lengths"])
            loss = loss_fn(logits, batch["labels"])
            if is_train:
                optimizer.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model_obj.parameters(), 5.0); optimizer.step()
        rows = int(batch["labels"].size(0))
        total_loss += float(loss.item()) * rows
        total_rows += rows
    return total_loss / max(total_rows, 1)

@torch.no_grad()
def predict_scores(model_obj: nn.Module, loader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    model_obj.eval()
    labels, scores = [], []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model_obj(batch["user_features"], batch["sequences"], batch["lengths"])
        labels.append(batch["labels"].detach().cpu().numpy())
        scores.append(logits.detach().cpu().numpy())
    return np.concatenate(labels), np.concatenate(scores)

def train_one_config(config: GRUContentConfig) -> tuple[GRUContentRecommender, dict[str, Any]]:
    seed_everything(SEED)
    model_obj = GRUContentRecommender(len(content_id_to_token), USER_FEATURE_DIM, config.embedding_dim, config.hidden_dim, config.dropout).to(DEVICE)
    optimizer = torch.optim.AdamW(model_obj.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    train_loader = make_loader(train_samples, config.batch_size, shuffle=True)
    valid_loader = make_loader(valid_samples, config.batch_size, shuffle=False)
    history, best_state, best_recall = [], None, -1.0
    for epoch in range(1, config.epochs + 1):
        started = time.perf_counter()
        train_loss = run_epoch(model_obj, train_loader, optimizer)
        valid_loss = run_epoch(model_obj, valid_loader)
        labels, scores = predict_scores(model_obj, valid_loader)
        valid_metrics = metric_at_k(labels, scores)
        row = {"epoch": epoch, "train_loss": train_loss, "valid_loss": valid_loss, "epoch_seconds": time.perf_counter() - started, **valid_metrics}
        history.append(row)
        print(json.dumps({**asdict(config), **row}, ensure_ascii=False))
        if valid_metrics["recall@10"] > best_recall:
            best_recall = valid_metrics["recall@10"]
            best_state = {k: v.detach().cpu().clone() for k, v in model_obj.state_dict().items()}
    if best_state is not None:
        model_obj.load_state_dict(best_state)
    return model_obj, {"config": asdict(config), "history": history, "best_valid_recall@10": best_recall}

EXPERIMENT_GRID = [
    GRUContentConfig(embedding_dim=16, hidden_dim=48, dropout=0.2, learning_rate=1e-3, batch_size=64, epochs=8),
    GRUContentConfig(embedding_dim=16, hidden_dim=64, dropout=0.2, learning_rate=1e-3, batch_size=64, epochs=8),
    GRUContentConfig(embedding_dim=32, hidden_dim=64, dropout=0.2, learning_rate=1e-3, batch_size=64, epochs=8),
]
RUN_FULL_TRAINING = False

if RUN_FULL_TRAINING:
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    experiment_results, best_model, best_result = [], None, None
    for config in EXPERIMENT_GRID:
        candidate_model, result = train_one_config(config)
        experiment_results.append(result)
        if best_result is None or result["best_valid_recall@10"] > best_result["best_valid_recall@10"]:
            best_model, best_result = candidate_model, result
    assert best_model is not None and best_result is not None
    test_loader = make_loader(test_samples, best_result["config"]["batch_size"], shuffle=False)
    test_labels, test_scores = predict_scores(best_model, test_loader)
    test_metrics = metric_at_k(test_labels, test_scores)
    torch.save({
        "model_state_dict": best_model.state_dict(), "model_class": "GRUContentRecommender",
        "vocab_size": len(content_id_to_token), "user_feature_dim": USER_FEATURE_DIM,
        "config": best_result["config"], "max_sequence_len": MAX_SEQUENCE_LEN,
        "pad_token_id": content_id_to_token[PAD_TOKEN], "unk_token_id": content_id_to_token[UNK_TOKEN],
        "feature_columns": FEATURE_COLUMNS,
    }, ARTIFACT_DIR / "best_shared_next_poi_gru.pt")
    with open(ARTIFACT_DIR / "content_id_vocab.json", "w", encoding="utf-8") as fp:
        json.dump({"content_id_to_token": content_id_to_token, "token_to_content_id": token_to_content_id, "pad_token": PAD_TOKEN, "unk_token": UNK_TOKEN}, fp, ensure_ascii=False, indent=2)
    poi_master.to_csv(ARTIFACT_DIR / "poi_master.csv", index=False, encoding="utf-8-sig")
    with open(ARTIFACT_DIR / "feature_encoder.pkl", "wb") as fp:
        pickle.dump(feature_encoder, fp)
    with open(ARTIFACT_DIR / "metrics.json", "w", encoding="utf-8") as fp:
        json.dump({"coverage": coverage, "split_sizes": split_sizes, "sample_sizes": sample_sizes, "baseline_valid": baseline_valid, "test_metrics": test_metrics, "experiments": experiment_results, "best_config": best_result["config"]}, fp, ensure_ascii=False, indent=2)
else:
    print("Training code is ready. Set RUN_FULL_TRAINING = True to start the full experiment.")
